# VSHORAD Tactical Training Pipeline

**Tier**: Tactical (L4 24GB / T4 16GB)

| Component | Model | Resolution | Epochs |
|-----------|-------|------------|--------|
| Detection | YOLOv8m | 960px | 120 |
| Classification | Swin-Small | 224px | 100 |

Lighter configuration for consumer-grade GPUs.
Works on Colab free tier (T4) or Colab Pro (L4).

## 1 Setup & Environment

In [ ]:
# 1.1 Installation

print(" Cleaning environment...")

!pip uninstall -y numpy
!pip uninstall -y numpy
!pip uninstall -y ultralytics wandb

print(" Installing...")

!pip install "numpy==1.26.4"
!pip install "opencv-python-headless<4.10"
!pip install "ultralytics==8.3.0" "timm==1.0.11" "codecarbon==2.4.2" "albumentations==1.4.18" "grad-cam==1.5.4" "lapx>=0.5.2" matplotlib seaborn pandas scikit-learn

print("\n" + "="*60)
print(" RESTART SESSION! Runtime -> Restart runtime")
print("  Then run from cell 1.2")
print("="*60)

In [ ]:
# 1.2 Diagnostics

import sys, os
import torch

def check(name, ok, ver=""):
  print(f"{'' if ok else ''} {name:<20} {ver}")

print(" AUDIT...\n")

print("--- LIBRARIES ---")
try:
  import numpy; check("NumPy", numpy.__version__.startswith("1"), numpy.__version__)
except: check("NumPy", False, "BRAK")
try:
  import ultralytics; check("Ultralytics", True, ultralytics.__version__)
except: check("Ultralytics", False, "BRAK")
try:
  import timm; check("Timm", True, timm.__version__)
except: check("Timm", False, "BRAK")
try:
  import cv2; check("OpenCV", True, cv2.__version__)
except: check("OpenCV", False, "BRAK")

print("\n--- GPU ---")
if torch.cuda.is_available():
  gpu = torch.cuda.get_device_name(0)
  mem = torch.cuda.get_device_properties(0).total_memory / 1e9
  print(f" {gpu} ({mem:.0f}GB)")
else:
  print(" No GPU available!")

print("\n--- DRIVE ---")
print(" Zamontowany" if os.path.exists('/content/drive/MyDrive') else " Niezamontowany")

In [ ]:
# 1.3 Imports

import os, sys, json, yaml, time, random, shutil, zipfile, warnings
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
from torchvision import datasets

import timm
from timm.data import Mixup
import albumentations as A
from albumentations.pytorch import ToTensorV2

from ultralytics import YOLO
from codecarbon import EmissionsTracker
from sklearn.metrics import accuracy_score

os.environ['WANDB_DISABLED'] = 'true'
os.environ['WANDB_MODE'] = 'disabled'
warnings.filterwarnings('ignore')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f" Imports OK | {DEVICE}")

In [ ]:
# 1.4 Google Drive

from google.colab import drive
drive.mount('/content/drive')
print(" Drive OK")

## 2 Configuration

In [ ]:
# 2.1 Paths

DRIVE_ROOT = Path('/content/drive/MyDrive/Inżynierka')
DATASETS_DIR = DRIVE_ROOT / 'Datasety Finalne' / 'Datasety_Zbalansowane'

ZIP_YOLO = DATASETS_DIR / 'Skydetect_Balanced_Yolo_Dataset.zip'
ZIP_SWIN = DATASETS_DIR / 'Skydetect_Balanced_Swin_Dataset.zip'

OUTPUT_ROOT = DRIVE_ROOT / 'Medium_components'
CHECKPOINTS_DIR = OUTPUT_ROOT / 'checkpoints'
YOLO_CHECKPOINT_DIR = CHECKPOINTS_DIR / 'yolo' / 'medium_yolov8m_960'
SWIN_CHECKPOINT_DIR = CHECKPOINTS_DIR / 'swin' / 'medium_swin_small_224'
LOGS_DIR = OUTPUT_ROOT / 'logs'
REPORTS_DIR = OUTPUT_ROOT / 'reports'

LOCAL_ROOT = Path('/content/datasets')
LOCAL_YOLO = LOCAL_ROOT / 'yolo'
LOCAL_SWIN = LOCAL_ROOT / 'swin'

for f in [OUTPUT_ROOT, CHECKPOINTS_DIR, YOLO_CHECKPOINT_DIR, SWIN_CHECKPOINT_DIR,
     LOGS_DIR, REPORTS_DIR, LOCAL_ROOT, LOCAL_YOLO, LOCAL_SWIN]:
  f.mkdir(parents=True, exist_ok=True)

print(f" Output: {OUTPUT_ROOT}")

In [ ]:
# 2.2 YOLO hyperparameters (Medium 960px)

gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0

# 960px requires more VRAM than 640px
if gpu_mem >= 20: # L4
  YOLO_BATCH = 16
elif gpu_mem >= 14: # T4
  YOLO_BATCH = 8
else:
  YOLO_BATCH = 4

YOLO_CONFIG = {
  'model': 'yolov8m.pt', 'task': 'detect',
  'epochs': 120, 'patience': 25, 'batch': YOLO_BATCH, 'imgsz': 960, 'device': 0,
  'optimizer': 'AdamW', 'lr0': 0.001, 'lrf': 0.01, 'momentum': 0.937,
  'weight_decay': 0.0005, 'warmup_epochs': 5,
  'box': 7.5, 'cls': 0.5, 'dfl': 1.5,
  'mosaic': 1.0, 'mixup': 0.1, 'degrees': 5.0, 'translate': 0.1,
  'scale': 0.5, 'shear': 2.0, 'fliplr': 0.5,
  'hsv_h': 0.015, 'hsv_s': 0.5, 'hsv_v': 0.4, 'erasing': 0.2,
  'workers': 4, 'project': str(CHECKPOINTS_DIR / 'yolo'),
  'name': 'medium_yolov8m_960', 'exist_ok': True, 'save_period': 10,
}

YOLO_CLASSES = {
  0: 'Fighter', 1: 'Helicopter', 2: 'Transport', 3: 'Bomber',
  4: 'Special', 5: 'UAV_Fixed', 6: 'UAV_Rotor', 7: 'Missile',
  8: 'Civilian', 9: 'Birds', 10: 'Decoys', 11: 'Unidentified'
}

print(f" YOLO: {YOLO_CONFIG['model']}, {YOLO_CONFIG['imgsz']}px, batch={YOLO_CONFIG['batch']} (auto dla {gpu_mem:.0f}GB)")

In [ ]:
# 2.3 Swin hyperparameters

SWIN_BATCH = 32 if gpu_mem >= 20 else 16

SWIN_CONFIG = {
  'model_name': 'swin_small_patch4_window7_224', 'img_size': 224,
  'num_classes': 56, 'pretrained': True,
  'epochs': 100, 'batch_size': SWIN_BATCH, 'num_workers': 4,
  'lr': 1e-4, 'weight_decay': 0.05,
  'warmup_epochs': 5, 'min_lr': 1e-6,
  'drop_path_rate': 0.2, 'label_smoothing': 0.1,
  'use_ema': True, 'ema_decay': 0.9999,
  'mixup_alpha': 0.8, 'cutmix_alpha': 1.0, 'mixup_prob': 0.5,
  'save_period': 5,
}

print(f" Swin: {SWIN_CONFIG['model_name']}, {SWIN_CONFIG['img_size']}px, batch={SWIN_CONFIG['batch_size']}")

## 3 Dataset Preparation

In [ ]:
# 3.1 Extract datasets

def unpack_if_needed(zip_path, target_dir):
  if not zip_path.exists():
    print(f" Missing: {zip_path}")
    return False
  if target_dir.exists() and any(target_dir.iterdir()):
    print(f" {target_dir.name} already extracted.")
    return True
  print(f" Extracting {zip_path.name}...")
  with zipfile.ZipFile(zip_path, 'r') as zf:
    zf.extractall(target_dir)
  print(" Gotowe!")
  return True

unpack_if_needed(ZIP_YOLO, LOCAL_YOLO)
unpack_if_needed(ZIP_SWIN, LOCAL_SWIN)

In [ ]:
# 3.2 Fix labels with 6 columns (remove confidence)

YOLO_DATASET_ROOT = Path('/content/datasets/yolo/Skydetect_Balanced_Yolo_Dataset')
LABELS_DIR = YOLO_DATASET_ROOT / 'labels'

print(" Searching and fixing labels with 6 columns...")

fixed_count = 0
error_files = []

for split in ['train', 'val', 'test']:
  split_dir = LABELS_DIR / split
  if not split_dir.exists():
    continue

  for label_file in tqdm(list(split_dir.glob('*.txt')), desc=f"Checking {split}"):
    try:
      with open(label_file, 'r') as f:
        lines = f.readlines()

      needs_fix = False
      fixed_lines = []

      for line in lines:
        parts = line.strip().split()
        if len(parts) == 6:
          # Has 6 columns - trim the last one (confidence)
          fixed_lines.append(' '.join(parts[:5]) + '\n')
          needs_fix = True
        elif len(parts) == 5:
          # OK - 5 kolumn
          fixed_lines.append(line)
        elif len(parts) > 0:
          # Other error - log
          error_files.append((label_file, len(parts)))
          fixed_lines.append(line)

      if needs_fix:
        with open(label_file, 'w') as f:
          f.writelines(fixed_lines)
        fixed_count += 1

    except Exception as e:
      error_files.append((label_file, str(e)))

print(f"\n Naprawiono {fixed_count} files z 6 kolumnami")

if error_files:
  print(f" {len(error_files)} files z innymi problemami:")
  for f, issue in error_files[:10]:
    print(f"  {f.name}: {issue}")
else:
  print(" No other issues!")

In [ ]:
# 3.3 Verify label fix

# Check random files after fix
print(" Verifying random files...\n")

import random

all_labels = list((LABELS_DIR / 'train').glob('*.txt'))
sample = random.sample(all_labels, min(5, len(all_labels)))

all_ok = True
for label_file in sample:
  with open(label_file, 'r') as f:
    lines = f.readlines()

  for i, line in enumerate(lines[:3]): # Pierwsze 3 linie
    parts = line.strip().split()
    if len(parts) != 5:
      print(f" {label_file.name} linia {i+1}: {len(parts)} kolumn")
      all_ok = False
    else:
      print(f" {label_file.name}: {parts[0]} {parts[1][:6]}... ({len(parts)} kolumn)")

print("\n" + (" Verification OK!" if all_ok else " Some files still have issues"))

In [ ]:
# 3.4 Path config

yolo_yaml = {
  'path': str(YOLO_DATASET_ROOT),
  'train': 'images/train', 'val': 'images/val', 'test': 'images/test',
  'names': YOLO_CLASSES
}

YOLO_CONFIG_PATH = LOCAL_ROOT / 'yolo_medium_config.yaml'
with open(YOLO_CONFIG_PATH, 'w') as f:
  yaml.dump(yolo_yaml, f)

SWIN_TRAIN_DIR = Path('/content/datasets/swin/Skydetect_Balanced_Swin_Dataset/train')
SWIN_VAL_DIR = Path('/content/datasets/swin/Skydetect_Balanced_Swin_Dataset/val')

print(f" YOLO config: {YOLO_CONFIG_PATH}")
print(f" Swin train: {SWIN_TRAIN_DIR}")

## 4 YOLO Training

YOLOv8m detection model (120 epochs).

In [ ]:
# 4.1 Checkpoint check

YOLO_LAST_PT = YOLO_CHECKPOINT_DIR / 'weights' / 'last.pt'
YOLO_BEST_PT = YOLO_CHECKPOINT_DIR / 'weights' / 'best.pt'
YOLO_RESUME = YOLO_LAST_PT.exists()

print(f" Resume: {YOLO_RESUME}" if YOLO_RESUME else "New training")

In [ ]:
# 4.2 Train YOLO

print(f" YOLO Medium: {YOLO_CONFIG['model']}, {YOLO_CONFIG['imgsz']}px, {YOLO_CONFIG['epochs']} epochs, batch={YOLO_CONFIG['batch']}")

yolo_tracker = EmissionsTracker(project_name="VSHORAD_YOLO_Medium", output_dir=str(LOGS_DIR), log_level='warning')

try:
  yolo_tracker.start()
  model = YOLO(str(YOLO_LAST_PT)) if YOLO_RESUME else YOLO(YOLO_CONFIG['model'])

  yolo_results = model.train(
    data=str(YOLO_CONFIG_PATH),
    epochs=YOLO_CONFIG['epochs'], patience=YOLO_CONFIG['patience'],
    batch=YOLO_CONFIG['batch'], imgsz=YOLO_CONFIG['imgsz'],
    device=YOLO_CONFIG['device'], optimizer=YOLO_CONFIG['optimizer'],
    lr0=YOLO_CONFIG['lr0'], lrf=YOLO_CONFIG['lrf'],
    momentum=YOLO_CONFIG['momentum'], weight_decay=YOLO_CONFIG['weight_decay'],
    warmup_epochs=YOLO_CONFIG['warmup_epochs'],
    box=YOLO_CONFIG['box'], cls=YOLO_CONFIG['cls'], dfl=YOLO_CONFIG['dfl'],
    mosaic=YOLO_CONFIG['mosaic'], mixup=YOLO_CONFIG['mixup'],
    degrees=YOLO_CONFIG['degrees'], translate=YOLO_CONFIG['translate'],
    scale=YOLO_CONFIG['scale'], shear=YOLO_CONFIG['shear'], fliplr=YOLO_CONFIG['fliplr'],
    hsv_h=YOLO_CONFIG['hsv_h'], hsv_s=YOLO_CONFIG['hsv_s'], hsv_v=YOLO_CONFIG['hsv_v'],
    erasing=YOLO_CONFIG['erasing'], workers=YOLO_CONFIG['workers'],
    project=YOLO_CONFIG['project'], name=YOLO_CONFIG['name'],
    exist_ok=True, pretrained=True, verbose=True, seed=42,
    save_period=YOLO_CONFIG['save_period'], resume=YOLO_RESUME,
  )

  yolo_emissions = yolo_tracker.stop()
  print(f"\n YOLO DONE! Carbon: {yolo_emissions:.4f} kg CO2")

except Exception as e:
  yolo_tracker.stop()
  print(f" ERROR: {e}")
  raise

## 5 Swin Training

Swin-Small classification model (100 epochs).

In [ ]:
# 5.1 SETUP SWIN

def get_train_transform(img_size=224):
  return A.Compose([
    A.LongestMaxSize(max_size=int(img_size * 1.2)),
    A.PadIfNeeded(img_size, img_size, border_mode=cv2.BORDER_CONSTANT, value=0),
    A.RandomCrop(img_size, img_size),
    A.HueSaturationValue(10, 30, 30, p=0.4),
    A.RandomBrightnessContrast(0.3, 0.3, p=0.4),
    A.HorizontalFlip(p=0.5),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
  ])

def get_val_transform(img_size=224):
  return A.Compose([
    A.LongestMaxSize(max_size=img_size),
    A.PadIfNeeded(img_size, img_size, border_mode=cv2.BORDER_CONSTANT, value=0),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
  ])

class SwinDataset(Dataset):
  def __init__(self, folder, transform):
    self.folder = folder
    self.transform = transform
  def __len__(self): return len(self.folder)
  def __getitem__(self, idx):
    path, label = self.folder.samples[idx]
    img = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
    return self.transform(image=img)['image'], label

train_folder = datasets.ImageFolder(str(SWIN_TRAIN_DIR))
val_folder = datasets.ImageFolder(str(SWIN_VAL_DIR))
SWIN_CLASSES = train_folder.classes
SWIN_CONFIG['num_classes'] = len(SWIN_CLASSES)

train_dataset = SwinDataset(train_folder, get_train_transform(SWIN_CONFIG['img_size']))
val_dataset = SwinDataset(val_folder, get_val_transform(SWIN_CONFIG['img_size']))

train_loader = DataLoader(train_dataset, batch_size=SWIN_CONFIG['batch_size'], shuffle=True, num_workers=4, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=SWIN_CONFIG['batch_size'], shuffle=False, num_workers=4, pin_memory=True)

print(f" Swin: {len(SWIN_CLASSES)} classes, {len(train_loader)} batchy")

In [ ]:
# 5.2 HELPERS

def update_ema(model, ema, decay=0.9999):
  with torch.no_grad():
    for ep, p in zip(ema.parameters(), model.parameters()):
      ep.data.mul_(decay).add_(p.data, alpha=1-decay)

def save_ckpt(model, ema, opt, sched, scaler, epoch, best, path):
  torch.save({'epoch': epoch, 'model': model.state_dict(), 'ema': ema.state_dict() if ema else None,
        'opt': opt.state_dict(), 'sched': sched.state_dict(), 'scaler': scaler.state_dict(),
        'best': best, 'classes': SWIN_CLASSES}, path)

def load_ckpt(path, model, ema=None, opt=None, sched=None, scaler=None):
  ckpt = torch.load(path, map_location=DEVICE)
  model.load_state_dict(ckpt['model'])
  if ema and ckpt.get('ema'): ema.load_state_dict(ckpt['ema'])
  if opt and ckpt.get('opt'): opt.load_state_dict(ckpt['opt'])
  if sched and ckpt.get('sched'): sched.load_state_dict(ckpt['sched'])
  if scaler and ckpt.get('scaler'): scaler.load_state_dict(ckpt['scaler'])
  return ckpt['epoch'], ckpt.get('best', 0)

@torch.no_grad()
def evaluate(model, loader, crit):
  model.eval()
  loss, preds, labels = 0, [], []
  for x, y in loader:
    x, y = x.to(DEVICE), y.to(DEVICE)
    out = model(x)
    loss += crit(out, y).item() * x.size(0)
    preds.extend(out.argmax(1).cpu().numpy())
    labels.extend(y.cpu().numpy())
  return loss/len(loader.dataset), accuracy_score(labels, preds)

print(" Helpers ready")

In [ ]:
# 5.3 Train Swin

SWIN_LAST = SWIN_CHECKPOINT_DIR / 'last.pth'
SWIN_BEST = SWIN_CHECKPOINT_DIR / 'best.pth'

print(f" Swin: {SWIN_CONFIG['model_name']}, {SWIN_CONFIG['epochs']} epochs")

swin = timm.create_model(SWIN_CONFIG['model_name'], pretrained=True, num_classes=SWIN_CONFIG['num_classes'], drop_path_rate=0.2).to(DEVICE)
ema = timm.create_model(SWIN_CONFIG['model_name'], pretrained=False, num_classes=SWIN_CONFIG['num_classes']).to(DEVICE)
ema.load_state_dict(swin.state_dict()); ema.eval()

opt = torch.optim.AdamW(swin.parameters(), lr=SWIN_CONFIG['lr'], weight_decay=0.05)
total_steps = len(train_loader) * SWIN_CONFIG['epochs']
warmup = len(train_loader) * 5
sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: s/warmup if s < warmup else 0.5*(1+np.cos(np.pi*(s-warmup)/(total_steps-warmup))))

crit = nn.CrossEntropyLoss(label_smoothing=0.1)
mixup = Mixup(mixup_alpha=0.8, cutmix_alpha=1.0, prob=0.5, switch_prob=0.5, mode='batch', label_smoothing=0.1, num_classes=SWIN_CONFIG['num_classes'])
scaler = GradScaler()

start, best = 0, 0
if SWIN_LAST.exists():
  start, best = load_ckpt(SWIN_LAST, swin, ema, opt, sched, scaler)
  start += 1
  print(f"  Resumed from {start}, best={best:.4f}")

tracker = EmissionsTracker(project_name="VSHORAD_Swin_Medium", output_dir=str(LOGS_DIR), log_level='warning')

try:
  tracker.start()
  for ep in range(start, SWIN_CONFIG['epochs']):
    swin.train()
    pbar = tqdm(train_loader, desc=f"Ep {ep+1}/{SWIN_CONFIG['epochs']}")
    for x, y in pbar:
      x, y = x.to(DEVICE), y.to(DEVICE)
      x, y = mixup(x, y)
      opt.zero_grad()
      with autocast(): loss = crit(swin(x), y)
      scaler.scale(loss).backward()
      scaler.step(opt); scaler.update(); sched.step()
      update_ema(swin, ema)
      pbar.set_postfix(loss=f"{loss.item():.4f}")

    val_loss, val_acc = evaluate(ema, val_loader, crit)
    print(f"  Val: loss={val_loss:.4f}, acc={val_acc:.4f}")

    if val_acc > best:
      best = val_acc
      save_ckpt(swin, ema, opt, sched, scaler, ep, best, SWIN_BEST)
      print(f"  New best: {best:.4f}")

    if (ep+1) % 5 == 0:
      save_ckpt(swin, ema, opt, sched, scaler, ep, best, SWIN_CHECKPOINT_DIR / f'ep{ep+1}.pth')
    save_ckpt(swin, ema, opt, sched, scaler, ep, best, SWIN_LAST)

  em = tracker.stop()
  print(f"\n SWIN DONE! Best={best:.4f}, Carbon={em:.4f}kg")
except Exception as e:
  tracker.stop()
  save_ckpt(swin, ema, opt, sched, scaler, ep if 'ep' in dir() else 0, best, SWIN_CHECKPOINT_DIR / 'emergency.pth')
  raise

## 6 Quick Evaluation

Post-training verification of both models.

In [ ]:
# 6.1 YOLO EVALUATION

from ultralytics import YOLO

YOLO_BEST_PT = YOLO_CHECKPOINT_DIR / 'weights' / 'best.pt'
YOLO_LAST_PT = YOLO_CHECKPOINT_DIR / 'weights' / 'last.pt'

yolo_weights = YOLO_BEST_PT if YOLO_BEST_PT.exists() else YOLO_LAST_PT
if yolo_weights.exists():
  print(f" Evaluating YOLO: {yolo_weights.name}")
  yolo_eval = YOLO(str(yolo_weights))
  yolo_metrics = yolo_eval.val(
    data=str(YOLO_CONFIG_PATH),
    imgsz=YOLO_CONFIG['imgsz'],
    batch=YOLO_CONFIG['batch'] // 2,
    device=0,
  )
  print(f"\n YOLO Results:")
  print(f"  mAP@0.5:   {yolo_metrics.box.map50:.4f}")
  print(f"  mAP@0.5:0.95: {yolo_metrics.box.map:.4f}")
  print(f"  Precision:   {yolo_metrics.box.mp:.4f}")
  print(f"  Recall:    {yolo_metrics.box.mr:.4f}")
else:
  print(" No YOLO weights found.")

In [ ]:
# 6.2 SWIN EVALUATION

SWIN_BEST_PTH = SWIN_CHECKPOINT_DIR / 'best.pth'
SWIN_LAST_PTH = SWIN_CHECKPOINT_DIR / 'last.pth'

swin_weights = SWIN_BEST_PTH if SWIN_BEST_PTH.exists() else SWIN_LAST_PTH
if swin_weights.exists() and val_loader:
  # Load best model
  checkpoint = torch.load(swin_weights, map_location=DEVICE)
  swin_eval = timm.create_model(
    SWIN_CONFIG['model_name'], pretrained=False,
    num_classes=SWIN_CONFIG['num_classes'],
  ).to(DEVICE)

  # Handle both checkpoint formats
  state_dict = (checkpoint.get('ema_state_dict') or checkpoint.get('ema')
         or checkpoint.get('model_state_dict') or checkpoint.get('model'))
  swin_eval.load_state_dict(state_dict)
  swin_eval.eval()

  # Top-1 accuracy
  criterion = nn.CrossEntropyLoss()
  val_loss, val_acc, _, _ = evaluate_swin(swin_eval, val_loader, criterion, DEVICE)

  # Top-5 accuracy
  top5_correct = 0
  with torch.no_grad():
    for images, labels in tqdm(val_loader, desc="Top-5 eval", leave=False):
      images, labels = images.to(DEVICE), labels.to(DEVICE)
      _, top5_preds = swin_eval(images).topk(5, dim=1)
      for i, label in enumerate(labels):
        if label in top5_preds[i]:
          top5_correct += 1
  top5_acc = top5_correct / len(val_loader.dataset)

  print(f"\n Swin Results:")
  print(f"  Top-1 Accuracy: {val_acc:.4f}")
  print(f"  Top-5 Accuracy: {top5_acc:.4f}")
  print(f"  Val Loss:    {val_loss:.4f}")
else:
  print(" No Swin weights found or no validation data.")

In [ ]:
# 6.3 SUMMARY

print("\n" + "=" * 70)
print(" TRAINING COMPLETE — SUMMARY")
print("=" * 70)

print(f"\n All outputs saved to: {OUTPUT_ROOT}")
print(f"\n Checkpoints:")
!ls -lh {YOLO_CHECKPOINT_DIR}/weights/ 2>/dev/null || echo "  YOLO: not found"
!ls -lh {SWIN_CHECKPOINT_DIR}/*.pth 2>/dev/null || echo "  Swin: not found"

print(f"\n Pipeline finished successfully!")